Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\1pasos_mlp_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 1
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 7)
Dimensiones de Y: (43788, 1)


In [9]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [10]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (43788, 84)


Se dividen nuevamente los conjuntos de datos

In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 84)
Las dimensiones de testX son:  (8801, 84)
Las dimensiones de valX son:  (4336, 84)


In [12]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [13]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])

    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

193/193 - 3s - 17ms/step - ia: 0.3077 - loss: 2.5183 - mae: 1.1409 - rmse: 1.5382 - smape: 1.4104 - val_ia: 0.2543 - val_loss: 0.4709 - val_mae: 0.5412 - val_rmse: 0.6356 - val_smape: 1.4046

Epoch 2/128                                           

193/193 - 0s - 2ms/step - ia: 0.3519 - loss: 1.5083 - mae: 0.9041 - rmse: 1.2031 - smape: 1.3753 - val_ia: 0.2714 - val_loss: 0.4140 - val_mae: 0.5065 - val_rmse: 0.5909 - val_smape: 1.3699

Epoch 3/128                                           

193/193 - 0s - 2ms/step - ia: 0.3924 - loss: 1.2410 - mae: 0.8170 - rmse: 1.0924 - smape: 1.3205 - val_ia: 0.2979 - val_loss: 0.3469 - val_mae: 0.4576 - val_rmse: 0.5383 - val_smape: 1.2343

Epoch 4/128                                           

193/193 - 0s - 2ms/step - ia: 0.4312 - loss: 1.0464 - mae: 0.7447 - rmse: 1.0045 - smape: 1.2590 - val_ia: 0.3346 - val_loss: 0.2914 - val_mae: 0.4136 - val_rmse: 0.4928 - val_smape: 1.1085

Epoch 5/128

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



25/25 - 3s - 134ms/step - ia: 0.6117 - loss: 0.4808 - mae: 0.4845 - rmse: 0.6753 - smape: 0.9477 - val_ia: 0.6135 - val_loss: 0.1972 - val_mae: 0.3260 - val_rmse: 0.4391 - val_smape: 0.8756

Epoch 2/16                                                                       

25/25 - 0s - 7ms/step - ia: 0.7846 - loss: 0.2137 - mae: 0.3156 - rmse: 0.4566 - smape: 0.6556 - val_ia: 0.6813 - val_loss: 0.1446 - val_mae: 0.2693 - val_rmse: 0.3775 - val_smape: 0.7353

Epoch 3/16                                                                       

25/25 - 0s - 7ms/step - ia: 0.8361 - loss: 0.1372 - mae: 0.2526 - rmse: 0.3630 - smape: 0.5557 - val_ia: 0.7163 - val_loss: 0.1189 - val_mae: 0.2400 - val_rmse: 0.3417 - val_smape: 0.6684

Epoch 4/16                                                                       

25/25 - 0s - 7ms/step - ia: 0.8556 - loss: 0.1041 - mae: 0.2205 - rmse: 0.3170 - smape: 0.5042 - val_ia: 0.7463 - val_loss: 0.0984 - val_mae: 0.2167 - val_rmse: 0.3099 - val_smape: 0

...

49/49 - 2s - 36ms/step - ia: 0.4118 - loss: 2.0312 - mae: 1.0768 - rmse: 1.4223 - smape: 1.2699 - val_ia: 0.4274 - val_loss: 0.5333 - val_mae: 0.5655 - val_rmse: 0.7087 - val_smape: 1.0659

Epoch 2/16                                                                         

49/49 - 0s - 3ms/step - ia: 0.4966 - loss: 1.4994 - mae: 0.9155 - rmse: 1.2146 - smape: 1.1561 - val_ia: 0.5014 - val_loss: 0.3695 - val_mae: 0.4610 - val_rmse: 0.5879 - val_smape: 0.9649

Epoch 3/16                                                                         

49/49 - 0s - 3ms/step - ia: 0.5369 - loss: 1.2176 - mae: 0.8272 - rmse: 1.0941 - smape: 1.0933 - val_ia: 0.5486 - val_loss: 0.2897 - val_mae: 0.4079 - val_rmse: 0.5203 - val_smape: 0.8945

Epoch 4/16                                                                         

49/49 - 0s - 3ms/step - ia: 0.5681 - loss: 1.0084 - mae: 0.7516 - rmse: 0.9959 - smape: 1.0476 - val_ia: 0.5955 - val_loss: 0.2195 - val_mae: 0.3459 - val_rmse: 0.4498 - val_sma

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



25/25 - 1s - 55ms/step - ia: 0.5748 - loss: 0.8404 - mae: 0.7036 - rmse: 0.8941 - smape: 1.0420 - val_ia: 0.7085 - val_loss: 0.1355 - val_mae: 0.2570 - val_rmse: 0.3615 - val_smape: 0.6895

Epoch 2/128                                                                        

25/25 - 0s - 4ms/step - ia: 0.6823 - loss: 0.4404 - mae: 0.5088 - rmse: 0.6685 - smape: 0.8704 - val_ia: 0.7352 - val_loss: 0.1155 - val_mae: 0.2423 - val_rmse: 0.3304 - val_smape: 0.7013

Epoch 3/128                                                                        

25/25 - 0s - 4ms/step - ia: 0.7293 - loss: 0.3228 - mae: 0.4346 - rmse: 0.5570 - smape: 0.7926 - val_ia: 0.7819 - val_loss: 0.0893 - val_mae: 0.1975 - val_rmse: 0.2887 - val_smape: 0.5829

Epoch 4/128                                                                        

25/25 - 0s - 4ms/step - ia: 0.7510 - loss: 0.2654 - mae: 0.3851 - rmse: 0.5149 - smape: 0.7245 - val_ia: 0.7968 - val_loss: 0.0816 - val_mae: 0.1845 - val_rmse: 0.2764 - val_sma

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



193/193 - 3s - 17ms/step - ia: 0.2181 - loss: 3.0360 - mae: 1.2057 - rmse: 1.6962 - smape: 1.6133 - val_ia: 0.2848 - val_loss: 0.8278 - val_mae: 0.6169 - val_rmse: 0.7381 - val_smape: 1.1665

Epoch 2/32                                                                         

193/193 - 0s - 2ms/step - ia: 0.2212 - loss: 2.8783 - mae: 1.1811 - rmse: 1.6629 - smape: 1.6079 - val_ia: 0.2846 - val_loss: 0.8114 - val_mae: 0.6102 - val_rmse: 0.7302 - val_smape: 1.1683

Epoch 3/32                                                                         

193/193 - 0s - 2ms/step - ia: 0.2181 - loss: 2.8733 - mae: 1.1808 - rmse: 1.6507 - smape: 1.6197 - val_ia: 0.2849 - val_loss: 0.7950 - val_mae: 0.6037 - val_rmse: 0.7224 - val_smape: 1.1705

Epoch 4/32                                                                         

193/193 - 0s - 2ms/step - ia: 0.2157 - loss: 2.8907 - mae: 1.1818 - rmse: 1.6563 - smape: 1.6195 - val_ia: 0.2854 - val_loss: 0.7789 - val_mae: 0.5974 - val_rmse: 0.7149 -

In [17]:
print(best)

{'activation': 1, 'batch': 2, 'dropout': 0.2, 'epochs': 4, 'layers': 1.0, 'learning_rate': 0.0002707756079796208, 'units': 4}


In [18]:
print(best)

{'activation': 1, 'batch': 2, 'dropout': 0.2, 'epochs': 4, 'layers': 1.0, 'learning_rate': 0.0002707756079796208, 'units': 4}
